In [0]:

-- 1. Parse streaming Kafka Orders
CREATE STREAMING LIVE VIEW stg_kafka_orders AS
SELECT
  from_json(
    message_payload, 
    'transaction_date STRING, stock_date STRING, product_id STRING, customer_id DOUBLE, store_id DOUBLE, transaction_mode STRING, quantity STRING'
  ) AS payload
FROM STREAM(mavenmarket.bronze.kafka_online_orders);

-- 2. Read historical ADLS transactions
CREATE STREAMING LIVE VIEW stg_adls_transactions AS
SELECT
  transaction_date, stock_date, product_id, customer_id, store_id, transaction_mode, quantity
FROM STREAM(mavenmarket.bronze.transactions);

-- 3. Union streams, normalize dates, and cast types
CREATE STREAMING LIVE VIEW stg_all_transactions_unified AS
SELECT
  COALESCE(
    try_to_date(transaction_date, 'MM/dd/yyyy'),
    try_to_date(transaction_date, 'yyyy-MM-dd'),
    try_to_date(transaction_date, 'dd-MMM-yy')
  ) AS transaction_date,
  COALESCE(
    try_to_date(stock_date, 'MM/dd/yyyy'),
    try_to_date(stock_date, 'yyyy-MM-dd'),
    try_to_date(stock_date, 'dd-MMM-yy')
  ) AS stock_date,
  CAST(product_id AS INT) AS product_id,
  CAST(customer_id AS INT) AS customer_id,
  CAST(store_id AS INT) AS store_id,
  transaction_mode,
  CAST(quantity AS INT) AS quantity
FROM (
  SELECT payload.* FROM STREAM(live.stg_kafka_orders)
  UNION ALL
  SELECT * FROM STREAM(live.stg_adls_transactions)
);

-- 4. Evaluate DQ rules into a boolean flag
CREATE STREAMING LIVE VIEW stg_transactions_with_dq AS
SELECT 
  *,
  (quantity > 0 AND product_id IS NOT NULL AND customer_id IS NOT NULL AND store_id IS NOT NULL AND transaction_date IS NOT NULL) AS passes_dq
FROM STREAM(live.stg_all_transactions_unified);

-- 5. The Clean Table (Uses actual columns for constraints so they resolve properly)
CREATE STREAMING LIVE TABLE silver_transactions (
  CONSTRAINT valid_quantity EXPECT (quantity > 0) ON VIOLATION DROP ROW,
  CONSTRAINT valid_product EXPECT (product_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_customer EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_store EXPECT (store_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_date EXPECT (transaction_date IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT "Clean unified transactions ready for Gold"
AS SELECT * EXCEPT (passes_dq) 
FROM STREAM(live.stg_transactions_with_dq);

-- 6. The Quarantine Table (Physically saves the bad rows for investigation)
CREATE STREAMING LIVE TABLE silver_transactions_quarantine
COMMENT "Quarantined transactions with data quality failures"
AS SELECT 
  * EXCEPT (passes_dq),
  current_timestamp() AS quarantine_timestamp
FROM STREAM(live.stg_transactions_with_dq)
WHERE passes_dq = FALSE;